In [1]:
# ==========================================
# 1. INSTALL AI/ML INFRASTRUCTURE
# ==========================================
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-cpu beautifulsoup4 sentence-transformers transformers
!echo "✅ LangChain, FAISS, and Hugging Face successfully installed!"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.5 MB/s eta 0:00:00
✅ LangChain, FAISS, and Hugging Face successfully installed!


In [10]:
# ==========================================
# 2. BUILD THE RAG PIPELINE
# ==========================================
import os
import warnings

# Set environment variables to resolve Colab terminal warnings
os.environ["USER_AGENT"] = "ColabUserAgent/1.0"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline

print("☁️ 1. Extracting complex research data from the web...")
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
loader = WebBaseLoader(url)
docs = loader.load()

print("✂️ 2. Chunking document into manageable context windows...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(docs)
print(f"   -> Created {len(splits)} text chunks.")

print("🧠 3. Initializing Embedding Model & Vector Database (FAISS)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("🤖 4. Booting up Local Open-Source LLM (TinyLlama)...")
# [THE ULTIMATE FIX]: We pivot to a modern Causal Decoder model (TinyLlama)
# This explicitly uses the "text-generation" task, permanently bypassing the Hugging Face Seq2Seq bugs.
hf_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    return_full_text=False # Crucial LangChain setting: prevents the LLM from repeating your question!
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# ==========================================
# 3. LANGCHAIN EXPRESSION LANGUAGE (LCEL)
# ==========================================
print("🔗 5. Assembling the Clean LCEL Pipeline...")

# Upgraded Prompt Template using modern Chat/Instruct tags for TinyLlama
template = """<|system|>
You are a highly intelligent AI assistant. Use the following context to answer the question concisely.
If the answer is not in the context, say "I don't know."
Context: {context}</s>
<|user|>
{question}</s>
<|assistant|>"""
custom_prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The core LCEL Data Flow
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)
print("✅ Pipeline Ready!\n")



☁️ 1. Extracting complex research data from the web...
✂️ 2. Chunking document into manageable context windows...
   -> Created 64 text chunks.
🧠 3. Initializing Embedding Model & Vector Database (FAISS)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🤖 4. Booting up Local Open-Source LLM (TinyLlama)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🔗 5. Assembling the Clean LCEL Pipeline...
✅ Pipeline Ready!



In [11]:
# ==========================================
# 4. EXECUTE THE PIPELINE
# ==========================================
print("==================================================")
user_query = "What is Task Decomposition in the context of LLM Agents?"
print(f"👤 USER QUERY: {user_query}")
print("==================================================")

# Invoke the entire chain with one command
response = rag_chain.invoke(user_query)

print("\n🤖 LLM ANSWER:")
print(response)
print("==================================================")

# Let's do a second query to prove dynamic retrieval
query_2 = "What are the components of the agent's memory?"
print(f"\n👤 USER QUERY 2: {query_2}")
print(f"🤖 LLM ANSWER: {rag_chain.invoke(query_2)}")

👤 USER QUERY: What is Task Decomposition in the context of LLM Agents?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🤖 LLM ANSWER:

Task Decomposition is a component of a LLM-powered autonomous agent system that helps the agent break down large tasks into smaller, manageable subgoals. By breaking down tasks, the agent can handle complex problems more efficiently and effectively. The task decomposition process involves identifying the subgoals that are required to achieve a larger goal, and then identifying the specific actions and outputs needed for each subgoal. This approach enables the agent to handle tasks that are not immediately apparent, which can be challenging for humans. Task Decomposition in the context of LLM Agents#
In LLM-powered autonomous agents, task decomposition is critical for efficient handling of complex problems. The agent receives a large, unstructured problem that requires multiple steps to accomplish, and the subgoals needed for each step are identified from the problem. Task decomposition is a key component of the LLM-powered agent's ability to handle complex problems effi